### RNN 언어 모델(RNNLM)
- ex. 'what will the fat cat sit on'
    - RNNLM 은 기본적으로 예측 과정에서 이전 시점의 출력을 현재 시점의 입력으로 사용
    - what -> will -> the -> fat -> cat -> sit -> on 순으로 예측됨
    - 훈련 과정에서는 위처럼 이전 시점의 예측 결과를 입력으로 넣는 것이 아니라 'wha will the fat cat sit' -> 'hat will the fat cat sit on' 로 예측하도록 훈련.
    - 이러한 RNN 훈련 기법을 교사 강요(Teacher Forcing)라고 함.

#### 교사 강요(Teacher Forcing)
- 테스트 과정에서 RNN 모델을 훈련시킬때 사용
- $t$ 시점에서 예측한 값을 $t+1$시점의 입력으로 사용하지 않고 $t$ 시점의 레이블 즉, 실제 알고 있는 정답을 입력으로 사용함.
- 잘못된 예측이 뒤의 예측까지 영향을 미칠 수 있으므로 교사 강요를 사용시 더 빠르고 효과적으로 훈련 가능.

#### RNNLM 구조
- 출력층의 활성화 함수는 소프트맥스 함수를 사용
1) t=1 시점에서 'what'이 입력으로 들어오면 RNNLM은 'will'이 다음에 올 단어로 가장 적절하다고 예측
    - 각 단어는 원-핫 벡터로 구현됨. will=(0,1,0,0,0,0,0), softmax 함수의 출력값이 (0.1, 0.6, 0.1, 0.1, 0.05, 0.05, 0) 이라고 하면 RNNLM은 'will'이 다음에 올 단어로 가장 적절하다고 예측
2) t=2 시점에서 'will'이 입력으로 들어오면 RNNLM은 'the'가 다음에 올 단어로 가장 적절하다고 예측
3) t=3 시점에서 'the'이 입력으로 들어오면 RNNLM은 'fat'이 다음에 올 단어로 가장 적절하다고 예측
4) t=4 시점에서 'fat'이 입력으로 들어오면 RNNLM은 'cat'이 다음에 올 단어로 가장 적절하다고 예측
    - <img src="img/zzn.png" width="600">
    - 임베딩층(embedding layer)은 단어의 원-핫 벡터를 밀집 벡터로 변환하는 층. NNLM에서 투사층(projection layer)이라고 불렸던 층과 동일한 역할을 함.
    - 단어 집합의 크기가 $V$일 때 임베딩 벡터의 크기를 $M$으로 설정하면 각 입력 단어들은 임베딩층에서 $V \times M$ 크기의 가중치 행렬과 곱해져서 M 차원의 밀집 벡터로 변환됨.
    - 임베딩층: $e_t = lookup(x_t)$
    - 은닉층: $h_t = \tanh(W_x e_t + W_{h}h_{t-1} + b_h)$
    - 출력층: $\hat{y_t} = softmax(W_y h_t + b_y)$


In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

text = """경마장에 있는 말이 뛰고 있다\n
그의 말이 법이다\n
가는 말이 고와야 오는 말이 곱다\n"""

tokenzier = Tokenizer()
tokenzier.fit_on_texts([text])
vocab_size = len(tokenzier.word_index) + 1 # 제로 패딩 고려
print('단어 집합의 크기:', vocab_size)
print(tokenzier.word_index)

sequences = []
for line in text.split('\n'):
    encoded = tokenzier.texts_to_sequences([line])[0]
    for i in range(1, len(encoded)):
        sequence = encoded[:i+1]
        sequences.append(sequence)

print('학습에 사용할 샘플의 개수:', len(sequences))
print(sequences)

max_len = max(len(l) for l in sequences) #모든 샘플에서 가장 길이가 긴 샘플의 길이
print('샘플의 최대 길이:', max_len)

sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(sequences)

단어 집합의 크기: 12
{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '곱다': 11}
학습에 사용할 샘플의 개수: 11
[[2, 3], [2, 3, 1], [2, 3, 1, 4], [2, 3, 1, 4, 5], [6, 1], [6, 1, 7], [8, 1], [8, 1, 9], [8, 1, 9, 10], [8, 1, 9, 10, 1], [8, 1, 9, 10, 1, 11]]
샘플의 최대 길이: 6
[[ 0  0  0  0  2  3]
 [ 0  0  0  2  3  1]
 [ 0  0  2  3  1  4]
 [ 0  2  3  1  4  5]
 [ 0  0  0  0  6  1]
 [ 0  0  0  6  1  7]
 [ 0  0  0  0  8  1]
 [ 0  0  0  8  1  9]
 [ 0  0  8  1  9 10]
 [ 0  8  1  9 10  1]
 [ 8  1  9 10  1 11]]


In [2]:
sequences = np.array(sequences)

X = sequences[:,:-1]
y = sequences[:, -1]
print(X)
print(y)

[[ 0  0  0  0  2]
 [ 0  0  0  2  3]
 [ 0  0  2  3  1]
 [ 0  2  3  1  4]
 [ 0  0  0  0  6]
 [ 0  0  0  6  1]
 [ 0  0  0  0  8]
 [ 0  0  0  8  1]
 [ 0  0  8  1  9]
 [ 0  8  1  9 10]
 [ 8  1  9 10  1]]
[ 3  1  4  5  1  7  1  9 10  1 11]


In [3]:
y = to_categorical(y, num_classes=vocab_size)
print(y)

[[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN

embedding_dim = 10
hidden_units = 32

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(SimpleRNN(hidden_units))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=2)

def sentence_generation(model, tokenizer, current_word, n):
    init_word = current_word
    sentence = ''
    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=5, padding='pre')

        result = model.predict(encoded, verbose=0) # 입력한 X에 대해 Y를 예측하고 Y를 result에 저장

        result = np.argmax(result, axis=1)

        for word, index in tokenizer.word_index.items():
            # 만약 예측한 단어와 인덱스가 동일한 단어가 있으면
            if index == result:
                break
        
        current_word = current_word + ' ' + word

        sentence = sentence + ' ' + word
    sentence = init_word + sentence

    return sentence


Train on 11 samples
Epoch 1/200
11/11 - 1s - loss: 2.4866 - accuracy: 0.0909
Epoch 2/200
11/11 - 0s - loss: 2.4718 - accuracy: 0.0909
Epoch 3/200
11/11 - 0s - loss: 2.4569 - accuracy: 0.1818
Epoch 4/200
11/11 - 0s - loss: 2.4418 - accuracy: 0.0909
Epoch 5/200
11/11 - 0s - loss: 2.4265 - accuracy: 0.0909
Epoch 6/200
11/11 - 0s - loss: 2.4108 - accuracy: 0.1818
Epoch 7/200
11/11 - 0s - loss: 2.3947 - accuracy: 0.2727
Epoch 8/200
11/11 - 0s - loss: 2.3781 - accuracy: 0.2727
Epoch 9/200
11/11 - 0s - loss: 2.3610 - accuracy: 0.1818
Epoch 10/200
11/11 - 0s - loss: 2.3433 - accuracy: 0.2727
Epoch 11/200
11/11 - 0s - loss: 2.3249 - accuracy: 0.3636
Epoch 12/200
11/11 - 0s - loss: 2.3058 - accuracy: 0.4545
Epoch 13/200
11/11 - 0s - loss: 2.2860 - accuracy: 0.4545
Epoch 14/200
11/11 - 0s - loss: 2.2654 - accuracy: 0.4545
Epoch 15/200
11/11 - 0s - loss: 2.2440 - accuracy: 0.4545
Epoch 16/200
11/11 - 0s - loss: 2.2218 - accuracy: 0.4545
Epoch 17/200
11/11 - 0s - loss: 2.1989 - accuracy: 0.4545
Epo

In [5]:
print(sentence_generation(model, tokenzier, '경마장에 ', 4))
print(sentence_generation(model, tokenzier, ' 그의 ', 2))
print(sentence_generation(model, tokenzier, '가는 ', 5))

경마장에  있는 말이 뛰고 있다
 그의  말이 법이다
가는  말이 고와야 오는 말이 곱다


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aashita/nyt-comments")

print("Path to dataset files:", path)

c:\Users\0627j\anaconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 480M/480M [00:49<00:00, 10.2MB/s] 

Extracting model files...


Path to dataset files: C:\Users\0627j\.cache\kagglehub\datasets\aashita\nyt-comments\versions\13


In [35]:
import pandas as pd
import numpy as np
from string import punctuation

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

df = pd.read_csv(path + "/ArticlesApril2018.csv")
print("열의 개수:", len(df.columns))
print(df.columns)

print(df['headline'].isnull().values.any())

headline = []
headline.extend(list(df.headline.values))
print(headline[:5])

print('총 샘플의 개수:', len(headline))
headline = [word for word in headline if word != "Unknown"]
print('노이즈 값 제거 후 샘플의 개수:', len(headline))

print(headline[:5])

def repreprocessing(raw_sentence):
    preprocessed_sentence = raw_sentence.encode('utf8').decode('ascii', 'ignore')
    # punctuation: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~ 
    return ''.join(word for word in preprocessed_sentence if word not in punctuation).lower()

preprocessed_headline = [repreprocessing(x) for x in headline]
print(preprocessed_headline[:5])

tokenzier = Tokenizer()
tokenzier.fit_on_texts(preprocessed_headline)
vocab_size = len(tokenzier.word_index) + 1
print('단어 집합의 크기:', vocab_size)

sequences = list()
for sentence in preprocessed_headline:
    encoded = tokenzier.texts_to_sequences([sentence])[0]
    for i in range(1, len(encoded)):
        sequence = encoded[:i+1]
        sequences.append(sequence)

print(sequences[:11])

index_to_word = {}
for key, value in tokenzier.word_index.items():
    index_to_word[value] = key

print('빈도수 상위 582번 단어:', index_to_word[582])

max_len = max(len(l) for l in sequences)
print('샘플의 최대 길이:', max_len)

sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(sequences[:3])

sequences = np.array(sequences)
X = sequences[:, :-1]
y = sequences[:, -1]
print(X[:3])

y = to_categorical(y, num_classes=vocab_size)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM

embedding_dim = 10
hidden_units = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(LSTM(hidden_units))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=2)

def sentence_generation(model, tokenizer, current_word, n):
    init_word = current_word
    sentence = ''
    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=max_len -1, padding='pre')

        result = model.predict(encoded, verbose=0)

        result = np.argmax(result, axis=1)
        for word, index in tokenizer.word_index.items():
            if index == result:
                break

        current_word = current_word + ' ' + word
        sentence = sentence + ' ' + word

    sentence = init_word + sentence
    return sentence
print(sentence_generation(model, tokenzier, 'i', 10))
print(sentence_generation(model, tokenzier, 'how', 10))

열의 개수: 15
Index(['articleID', 'articleWordCount', 'byline', 'documentType', 'headline',
       'keywords', 'multimedia', 'newDesk', 'printPage', 'pubDate',
       'sectionName', 'snippet', 'source', 'typeOfMaterial', 'webURL'],
      dtype='object')
False
['Former N.F.L. Cheerleaders’ Settlement Offer: $1 and a Meeting With Goodell', 'E.P.A. to Unveil a New Rule. Its Effect: Less Science in Policymaking.', 'The New Noma, Explained', 'Unknown', 'Unknown']
총 샘플의 개수: 1324
노이즈 값 제거 후 샘플의 개수: 1214
['Former N.F.L. Cheerleaders’ Settlement Offer: $1 and a Meeting With Goodell', 'E.P.A. to Unveil a New Rule. Its Effect: Less Science in Policymaking.', 'The New Noma, Explained', 'How a Bag of Texas Dirt  Became a Times Tradition', 'Is School a Place for Self-Expression?']
['former nfl cheerleaders settlement offer 1 and a meeting with goodell', 'epa to unveil a new rule its effect less science in policymaking', 'the new noma explained', 'how a bag of texas dirt  became a times tradition', 'is s

In [38]:
def sentence_generation(model, tokenizer, current_word, n):
    init_word = current_word
    sentence = ''
    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=max_len -1, padding='pre')

        result = model.predict(encoded, verbose=0)

        result = np.argmax(result, axis=1)
        for word, index in tokenizer.word_index.items():
            if index == result:
                break

        current_word = current_word + ' ' + word
        sentence = sentence + ' ' + word

    sentence = init_word + sentence
    return sentence
print(sentence_generation(model, tokenzier, 'i', 10))
print(sentence_generation(model, tokenzier, 'how', 10))

i want to be rich and im not sorry its life
how to make a crossword puzzle bone to help a genius
